In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


In [ ]:
# Reading the saved data pickle file
rng = pd.date_range('2019-02-24', periods=661, freq='T')
df  = pd.DataFrame()
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/sentiment.pkl', compression=None)
df_companies=pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/press_release_companies_statistics.pkl', compression=None)

In [ ]:
df_companies

In [ ]:
df["articles"].describe()
df = df.join(df_sentiment)
df[["VIX"]] = df["adj close"].apply(np.int)
df["log_VIX"]=np.log(df["VIX"]/df["VIX"].shift(1))
df[['sentiment']] = df[["sentiment"]].apply(pd.to_numeric)
df["sentiment_ewm"]=df["sentiment"].ewm(20).mean()
df['vix_ln_return'] = np.log(df['VIX']/df['VIX'].shift(1))
df['vix_ln_return_ewm100']=df['vix_ln_return'].ewm(span=100).mean()#.plot(title='vix ln returns')
df['date'] = pd.date_range(start='25/9/2019', periods=len(df), freq='D')
df_companies['count'] = df_companies['count'].apply(pd.to_numeric)

In [ ]:
df

In [ ]:
df_companies.plot(kind="hist")

In [ ]:
cols = (df_companies[["companies", "count"]]
        .groupby("companies")
        .mean()
        .sort_values(by="count", ascending=False)[:150].index)
# best_features = df_companies.loc[feature_importance_df.Feature.isin(cols)]

plt.figure(figsize=(20,150))
chart=sns.barplot(x="count", y="companies", data=df_companies.sort_values(by="count",ascending=False))
chart.bar_label(chart.containers[0])
plt.title('S&P500 Companies by Press Release Count')
plt.tight_layout()
plt.savefig('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/S&P500 Companies by Press Release Count.png')